# iterative_probe — 変調範囲 × GroupDRO step size

変調範囲（`fc` / `stage4+fc`）× GroupDRO step size（1e-3 / 1e-2）の 4 run と、比較対象の
通常 ResNet を読む。seed 42、warmup 2 epoch → stage01 5 epoch → stage02 5 epoch、cohort 10 クラスタ。
run-id と条件の対応は [runs.md](runs.md)。

この notebook は **分析の問い・可視化・主張と、その根拠の表示**を持つ。集計と群の定義は script 側にあり、
図と表は次の順に作る。上から順に 1 回ずつ実行すれば、結果表と notebook 内の図が揃う。

```bash
uv run python analysis/common/predictions.py --study iterative_probe --split test \
  --run-dir analysis/iterative_probe/runs/20260921T103036Z-resnet-chexpert-s42-5538 \
  --run-dir analysis/iterative_probe/runs/20260921T103222Z-iterative-s42-d24d \
  --run-dir analysis/iterative_probe/runs/20260921T103219Z-iterative-s42-e992 \
  --run-dir analysis/iterative_probe/runs/20260921T103225Z-iterative-s42-3166 \
  --run-dir analysis/iterative_probe/runs/20260921T103225Z-iterative-s42-fac5
uv run python analysis/iterative_probe/collect.py \
  20260921T103222Z-iterative-s42-d24d 20260921T103219Z-iterative-s42-e992 \
  20260921T103225Z-iterative-s42-3166 20260921T103225Z-iterative-s42-fac5 \
  --baseline 20260921T103036Z-resnet-chexpert-s42-5538
uv run python analysis/iterative_probe/groups.py --split test
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.style
import pandas as pd
import rootutils
from IPython.display import display

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)
from analysis.common.paths import STYLE_SHEET  # noqa: E402, I001
from analysis.iterative_probe.plots import (  # noqa: E402
    GROUP_DRO_PANELS,
    against_baseline,
    all_cohorts,
    epoch_figure,
    fairness_metrics,
    fairness_ranges,
    intersection_groups,
    global_performance,
)

matplotlib.style.use(STYLE_SHEET)
PACKAGE = ROOT / "analysis" / "iterative_probe"
RESULTS = PACKAGE / "results"
SPLIT = "test"
MODULATIONS = ["fc", "stage4+fc"]

frame = pd.read_csv(RESULTS / "epoch_metrics.csv")
baseline = pd.read_csv(RESULTS / "baseline_epoch_metrics.csv")
cohorts = pd.read_csv(RESULTS / "cohort_groups.csv")
groups = pd.read_csv(RESULTS / f"group_metrics_{SPLIT}.csv")
frame.groupby(["modulation", "step_size"])["run_id"].agg(["first", "count"])

## 指標の読み方

hidden cohort 系の指標は名前が似ているので、`hidden_cohort_logger.py` の定義で揃えておく。

| 列 | 定義 | 図の見出し |
|---|---|---|
| `val/hidden_min_auroc` | 10 cohort の AUROC の**最小**。いちばん悪い群の性能 | Worst-group AUROC |
| `val/hidden_auroc_gap` | 同じく**最大 − 最小**。群間がどれだけ開いているか | Cohort AUROC spread |
| `val/hidden_loss_gap` | cohort 平均 loss の最大 − 最小 | （図にしていない） |

spread は worst group の値ではなく**散らばりの幅**なので、小さいほど群間が揃っている。
worst が下がっても best がもっと下がれば spread は縮むため、2 つは別々に読む。

**cohort は stage ごとに、さらに run ごとに引き直される。** hidden group は run 間でも stage 間でも
別物なので、群の同一性を前提にした読み方はできない。

## 到達点

各 run の最終 epoch。`hidden_*` は cohort が存在する stage にだけあるので、warmup では空になる。

In [ ]:
COLUMNS = {
    "val/auroc": "global AUROC",
    "val/bacc": "global bACC",
    "val/hidden_min_auroc": "worst AUROC",
    "val/hidden_min_bacc": "worst bACC",
    "val/hidden_auroc_gap": "cohort AUROC spread",
    "train/group_dro/weight_entropy": "weight entropy",
    "train/group_dro/max_q": "max q",
}

last = frame.sort_values("run_epoch").groupby(["modulation", "step_size"]).tail(1)
last = last.set_index([last["modulation"], last["step_size"]])[list(COLUMNS)].rename(columns=COLUMNS)
last.index.names = ["modulation", "step size"]
last.round(4)

## 学習曲線

横軸は run 全体の通し epoch。縦の区切りは stage の境目で、そこで cohort が引き直され、
GroupDRO の `q` も初期化される。seed は 42 の 1 本だけなので、seed 間のばらつき帯は引けない。

図は変調範囲ごとに分け、線の色は step size だけを表す。4 条件を 1 枚に重ねると線が交差して読めない。

In [ ]:
matplotlib.style.use(STYLE_SHEET)
panels = [
    ("val/loss", "Validation loss"),
    ("val/auroc", "Validation AUROC"),
    ("val/bacc", "Validation balanced accuracy"),
    ("val/hidden_min_auroc", "Worst-group AUROC"),
    ("val/hidden_min_bacc", "Worst-group balanced accuracy"),
    ("val/hidden_auroc_gap", "Cohort AUROC spread"),
]
step_colors = {0.001: "#0173B2", 0.01: "#DE8F05"}

for modulation in MODULATIONS:
    figure, axes = plt.subplots(
        2,
        3,
        figsize=(13, 7),
        squeeze=False,
        constrained_layout=False,
    )
    figure.subplots_adjust(top=0.72, bottom=0.10, hspace=0.65, wspace=0.28)

    data = frame[frame["modulation"] == modulation]

    for axis, (column, title) in zip(axes.ravel(), panels, strict=True):
        for step, series in data.groupby("step_size"):
            axis.plot(series["run_epoch"], series[column], color=step_colors[step], label=f"step {step:g}")
        axis.set(title=title, xlabel="Epoch")
    handles, labels = axes.ravel()[0].get_legend_handles_labels()
    figure.suptitle(f"modulation: {modulation}", y=0.995)
    figure.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.90),
        ncol=len(labels),
    )
    plt.show()

## baseline — 通常の ResNet と並べる

反復で何を失っているのかは、反復同士を比べても出てこない。比較対象は
`20260921T103036Z-resnet-chexpert-s42-5538`（ERM、30 epoch）で、
[initial_resnet_vs_invariant](../initial_resnet_vs_invariant/) が invariant 化の対照に使う run と同じもの。
`data_manifest.json` の train / val の sha256、optimizer（AdamW lr 1e-4 / wd 0.01）、batch size 128、
class weight `[0.2014, 1.7986]`、seed 42、transform が iterative 側と一致するので、
**global の val 指標はそのまま並べて読める**。

揃っていないのは学習の中身のほうになる。baseline は ResNet-50 全体を 30 epoch の ERM、iterative は
Spatial LoRA + metadata 条件付けを載せて warmup 2 epoch のあと cohort GroupDRO へ切り替える 12 epoch。
backbone を凍結していないので、パラメータ数では baseline の上位集合になる。

baseline は 3 点で見る。iterative と同じ epoch 予算での位置、checkpoint に選ばれる最良、30 epoch 後。
同じ epoch 数で比べるだけでは、baseline がまだ伸びる途中なのか収束済みなのかが読めない。

In [ ]:
pd.read_csv(RESULTS / "global_comparison.csv", index_col="run").round(4)

In [ ]:
for modulation in MODULATIONS:
    figure = against_baseline(frame, baseline, modulation)
    display(figure)
    plt.close(figure)

### 全体性能の snapshot 比較

学習曲線の推移に加えて、baseline の同一 epoch 予算・best AUROC・final と、iterative の同一予算時点を
AUROC と balanced accuracy で並べる。baseline と iterative の学習条件は異なるため、これは到達点の
記述的な比較として読む。

In [ ]:
comparison = pd.read_csv(RESULTS / "global_comparison.csv", index_col="run")
figure = global_performance(comparison)
display(figure)
plt.close(figure)

## 公平性 — demographic 群ごと

ここまでの `val/<属性>/worst_group_auroc` には、属性ごとの worst と gap しか残らない。
群そのものの性能も交差群も run artifact に無いので、予測 cache から群別指標を作る
（切り方と理由は `groups.py` の docstring）。

評価は **test split** で行う。各 run の checkpoint は val AUROC で選んでいるので、val で群別の性能を
比べると選択の効いた側に寄る。test は 5 model のどれも見ていない。**この節の global 値が上の節と
一致しないのは、上が val でここが test だから**で、race の絞り込み（White / Asian / Black）も効いている。

In [ ]:
summary = pd.read_csv(RESULTS / f"fairness_summary_{SPLIT}.csv", index_col=["grouping", "model"])
summary.round(4)

gap だけでは worst が上がったのか best が下がったのかが読めないので、worst と best の値も並べている。
`Eopp1` は群間の TPR の最大差、`Eopp0` は TNR の最大差、`Eodds` は `(TPR gap + FPR gap) / 2` で、
`projects/*/utils/metrics.py` と同じ定義になる（単独属性で一致することは `groups.py` が検算する）。

1 枚目は粒度ごとの worst → best の幅。線が短いほど群間が揃っていて、線の位置が全体の水準になる。
2 枚目は 3 属性の交差 12 群を 1 群 1 行で並べる。`n` の小さい群は AUROC も TPR も揺れるので、
値を読む前に群名に添えた `n` を見る。

In [ ]:
for modulation in MODULATIONS:
    figure = fairness_ranges(groups, modulation, SPLIT)
    display(figure)
    plt.close(figure)
    figure = intersection_groups(groups, modulation, SPLIT)
    display(figure)
    plt.close(figure)

### Eopp0 / Eopp1 / Eodds

群別の予測率の差も、粒度ごとに比較する。`Eopp1` は TPR（正例を正しく陽性と予測する率）の最大差、
`Eopp0` は TNR（負例を正しく陰性と予測する率）の最大差、`Eodds` は TPR gap と FPR gap の平均である。
いずれも 0 に近いほど公平性の gap が小さいが、性能水準そのものではないため、直前の AUROC / bACC と併読する。
単独属性から age × sex × race の交差群までを同じ test 母集団で集計している。

In [ ]:
fairness_columns = ["Eopp0", "Eopp1", "Eodds"]
summary[fairness_columns].round(4)

for modulation in MODULATIONS:
    figure = fairness_metrics(summary, modulation, SPLIT, groups)
    display(figure)
    plt.close(figure)

## GroupDRO が動いたか

`weight_entropy` の上限は一様分布の log(10) = 2.3026。ここから離れるほど、特定の cohort へ
重みが寄っている。判定は「stage 内で寝るか（均衡）、下がり続けるか（未収束）」で行う。

In [ ]:
for modulation in MODULATIONS:
    figure = epoch_figure(frame, modulation, GROUP_DRO_PANELS, columns=2)
    display(figure)
    plt.close(figure)

## q は何に寄ったのか

cohort は stage ごとに引き直されるので、群を stage 間で追うことはできない。代わりに
**分析単位を `(run, stage, cohort)` とし、stage の中だけで `q` が何と相関するかを見る**。
知りたいのは「群 k がどうなったか」ではなく「DRO が何を難しさと見なしたか」なので、
stage 内で閉じた問いとして立てられる。4 run × 2 stage = 8 回の再抽選がそのまま反復サンプルになる。

対立仮説は 3 つ。

1. **重み由来** — `q` が陽性 class weight の大きい群に寄る。`weighting=inverse` では class weight が
   群の陽性率の決定的な関数なので、これは「陽性率の低い群に寄る」と同義
2. **難しさ由来** — `q` が AUROC の低い群に寄る。想定どおり
3. **サイズ由来** — `q` が小さい群に寄る。少数群は loss の分散が大きい

In [ ]:
pd.read_csv(RESULTS / "cohort_correlations.csv").round(3)

## 全 cohort の推移

区分（top3 / bottom3 など）で切ると epoch ごとの順位変動が潰れるので、10 群すべてを描く。

群の色は **stage 最終 epoch の `q` の順位**で決める（濃いほど `q` が大きい）。同じ図の中で同じ色は
同じ群を指すので、`q` の panel で上に行く線が AUROC の panel でどう動くかを追える。順位そのものは
色が表すので凡例は置かない。stage をまたぐと cohort が変わるため、色の対応も stage 内で閉じる。

In [ ]:
for modulation in MODULATIONS:
    for step in sorted(frame["step_size"].unique()):
        figure = all_cohorts(frame, cohorts, modulation, step)
        display(figure)
        plt.close(figure)

---

読み取った内容と、そこから決めた次の実験方針は `reports/` に書く。対象 run の一覧と状態は
[runs.md](runs.md) が持つ。